# Huấn Luyện Mô Hình ConFit v2 (Findings of ACL 2025) cho BTL NLP
### Đề tài: Khớp Hồ Sơ CV & Tin Tuyển Dụng JD (Person-Job Fit) trên Flow-Cortex

Notebook này thực thi toàn bộ pipeline huấn luyện từ đầu đến cuối theo bài báo:
1. **Chuẩn bị Dữ liệu:** Nạp dataset benchmark `cnamuangtoun/resume-job-description-fit` (HuggingFace) + tập tin IT song ngữ Việt - Anh, thực hiện **Group Split 80/10/10 theo Resume ID** (0% Data Leakage).
2. **HYRE (Hypothetical Resume Embedding):** Sinh CV giả định lý tưởng từ JD bằng LLM để xóa bỏ độ lệch văn phong giữa JD và CV.
3. **RUM (Runner-Up Hard-Negative Mining):** Khai thác mẫu âm khó trong dải á quân (Top 3% - 4%) nhằm loại bỏ False Negatives.
4. **Huấn luyện Bi-Encoder + InfoNCE Loss + PEFT LoRA:** Tối ưu hóa trên 1 GPU T4 (khoảng 25 - 40 phút).
5. **Đánh giá Thực nghiệm (Ablation Study):** Xuất bảng so sánh BM25 vs Zero-shot vs In-Batch vs ConFit v2.
6. **Đóng gói ONNX INT8:** Xuất mô hình gọn nhẹ (~280 MB) sẵn sàng chạy trực tiếp trên CPU của container Flow-Cortex (`< 150 ms`).

## 1. Cài đặt Môi trường & Kiểm tra GPU

In [ ]:
# Kiểm tra GPU được cấp phát (Nên chọn Runtime: GPU T4)
!nvidia-smi

# Cài đặt các thư viện cần thiết
!pip install -q torch transformers datasets peft accelerate faiss-cpu pandas numpy scikit-learn scipy tqdm onnx onnxruntime tabulate

## 2. Chuẩn bị Dữ liệu (HuggingFace + IT Song ngữ Việt - Anh)

In [ ]:
# Tải mã nguồn hoặc chạy trực tiếp prepare_datasets.py
!python data/prepare_datasets.py --output_dir ./data/processed --max_samples 8000

## 3. Sinh CV Giả Định (HYRE - Hypothetical Resume Embedding)

In [ ]:
# Chạy HYRE Generator (Có thể dùng provider gemini / openai nếu có key, hoặc chế độ mock/heuristic tự động)
import os
# os.environ["GEMINI_API_KEY"] = "your_api_key_here"
!python hyre/generate_hyre.py --data_dir ./data/processed --provider mock

## 4. Huấn luyện Vòng 1 (Warmup Model với In-Batch Negatives)

In [ ]:
!python train.py \
    --data_dir ./data/processed \
    --pretrained_model intfloat/multilingual-e5-base \
    --output_dir ./checkpoints/warmup \
    --epochs 3 \
    --batch_size 8 \
    --lr 2e-5 \
    --use_lora

## 5. Khai thác Mẫu Âm Khó Á Quân (RUM - Runner-Up Mining)

In [ ]:
!python mining/mine_runner_up.py \
    --data_dir ./data/processed \
    --model_name_or_path ./checkpoints/warmup/best_model \
    --output_file ./data/processed/hard_negatives.json \
    --lower 0.04 \
    --upper 0.03

## 6. Huấn luyện Chính thức ConFit v2 (Full: HYRE + RUM)

In [ ]:
!python train.py \
    --data_dir ./data/processed \
    --pretrained_model intfloat/multilingual-e5-base \
    --hard_negatives ./data/processed/hard_negatives.json \
    --output_dir ./checkpoints/confit_v2_full \
    --epochs 5 \
    --batch_size 8 \
    --lr 2e-5 \
    --use_lora

## 7. Đánh giá Thực nghiệm & Xuất Bảng Ablation Study cho Báo cáo NLP

In [ ]:
!python evaluate.py \
    --data_dir ./data/processed \
    --model_path ./checkpoints/confit_v2_full/best_model \
    --baseline_model intfloat/multilingual-e5-base

## 8. Đóng gói Mô hình sang ONNX INT8 cho Flow-Cortex

In [ ]:
!python export_onnx.py \
    --model_path ./checkpoints/confit_v2_full/best_model \
    --output_dir ./confit_v2_onnx_export

# Tải mô hình ONNX về máy để tích hợp vào container media-worker
from google.colab import files
files.download('./confit_v2_onnx_export/confit_v2_int8.onnx')